# 03 — Llama-3.2-3B-Instruct: baseline + LoRA fine-tuning

Pipeline (all under the 8B-parameter limit, **3.2B params**):

1. **Baseline**: score the base model on `val` and `test`, 0-shot and few-shot.
2. **Fine-tune**: QLoRA (4-bit base + LoRA adapters) on `train`. After every epoch it measures `val` accuracy and keeps the best adapter.
3. **Final eval**: best fine-tuned adapter on `val` and `test`, compared side by side with the baseline.

**How answers are scored:** instead of generating text, the model's next-token probabilities for `1`, `2`, `3`, `4` are compared and the highest wins. So every question gets a valid answer, with no parsing.

**Hardware:** needs an NVIDIA GPU. A free **Colab T4** or **Kaggle T4/P100** is enough. Expect roughly 30–45 min end to end on a T4.

**Before running:** accept the licence at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and create a token at https://huggingface.co/settings/tokens. Then add it as a secret named `HF_TOKEN` (Colab: key icon in the left sidebar; Kaggle: Add-ons → Secrets).

**Private GitHub repo:** also add a secret named `GITHUB_TOKEN`. Create it at https://github.com/settings/personal-access-tokens: choose "Fine-grained token", select only this repository, and grant *Contents: Read-only*.

## 1. Environment

In [ ]:
import os, sys, subprocess
from pathlib import Path

GH_USER = "Imalsha-Ranasinghe"
REPO    = f"github.com/{GH_USER}/SinhalaMMLU-Shared-Task.git"

def get_secret(name):
    # Read a secret from env vars, Colab secrets or Kaggle secrets
    if os.getenv(name):
        return os.getenv(name).strip()
    try:
        from google.colab import userdata
        return userdata.get(name).strip()
    except Exception as e:
        print(f"[{name}] not read from Colab secrets: {type(e).__name__}: {e}")
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name).strip()
    except Exception:
        return None

clone_dir = Path.cwd() / "SinhalaMMLU-Shared-Task"
if (Path.cwd() / "src").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    ROOT = Path.cwd().parent
elif (clone_dir / "src").exists():          # already cloned earlier in this session
    ROOT = clone_dir
else:
    # Fresh Colab / Kaggle session: clone the project. Private repo -> needs a GITHUB_TOKEN secret.
    gh_token = get_secret("GITHUB_TOKEN")
    print("GITHUB_TOKEN found:", bool(gh_token))
    auth = f"{GH_USER}:{gh_token}@" if gh_token else ""
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}{REPO}", str(clone_dir)],
                       capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        err = r.stderr.replace(gh_token, "***") if gh_token else r.stderr
        raise RuntimeError("git clone failed: " + err)
    # don't leave the token in .git/config
    subprocess.run(["git", "-C", str(clone_dir), "remote", "set-url", "origin", f"https://{REPO}"])
    ROOT = clone_dir

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-hf.txt"], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. On Colab: Runtime -> Change runtime type -> T4 GPU"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Hugging Face login (Llama 3.2 is a gated model)
from huggingface_hub import login

token = get_secret("HF_TOKEN")
login(token=token) if token else login()

## 2. Configuration — edit this cell

In [ ]:
MODEL_ID     = "meta-llama/Llama-3.2-3B-Instruct"   # ungated copy with identical weights: "unsloth/Llama-3.2-3B-Instruct"
SHORT_NAME   = "llama3.2-3b-instruct"
MAX_PARAMS   = 8e9            # project limit
LOAD_IN_4BIT = True
SEED         = 42

EVAL_SPLITS    = ["val", "test"]
BASELINE_SHOTS = [0, 3]       # few-shot settings for the base model (examples come from train, same subject)
SCORER_TOKENS  = 8000         # tokens per forward pass at eval time; lower it if you hit CUDA OOM

# --- LoRA fine-tuning ---
DO_TRAIN     = True
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EPOCHS       = 3
LR           = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4              # effective batch = 16
MAX_LEN      = 1280           # longest 0-shot training example is ~1250 tokens

RESULTS_DIR = ROOT / "results"
ADAPTER_DIR = ROOT / "outputs" / f"{SHORT_NAME}-lora"
print("Adapter will be saved to", ADAPTER_DIR)

## 3. Parameter budget check

In [ ]:
from accelerate import init_empty_weights
from transformers import AutoConfig, AutoModelForCausalLM

with init_empty_weights():
    _m = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(MODEL_ID))
n_params = sum(p.numel() for p in _m.parameters())   # tied embeddings counted once
del _m
print(f"{MODEL_ID}: {n_params / 1e9:.2f}B parameters (limit {MAX_PARAMS / 1e9:.0f}B)")
assert n_params < MAX_PARAMS

## 4. Load data and model

In [ ]:
from src.data import load_jsonl

data = {s: load_jsonl(ROOT / "data" / "splits" / f"{s}.jsonl") for s in ["train", "val", "test"]}
for s, d in data.items():
    print(f"{s:5s}: {len(d)}")

In [ ]:
from src.hf_model import load_model_and_tokenizer, ChoiceScorer, render_prompt
from src.prompting import build_messages

model, tok = load_model_and_tokenizer(MODEL_ID, load_in_4bit=LOAD_IN_4BIT)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print(render_prompt(tok, build_messages(data["test"].iloc[0].to_dict())))

## 5. Baseline (no fine-tuning)

In [ ]:
import json
import pandas as pd
from src.prompting import ShotSampler
from src.evaluate import run_eval, compute_metrics, save_metrics

def evaluate(scorer, run_name, shots, splits=EVAL_SPLITS):
    sampler = ShotSampler(data["train"], k=shots, seed=SEED)
    run_dir = RESULTS_DIR / run_name
    rows = {}
    for split_name in splits:
        preds = run_eval(data[split_name], scorer, sampler, run_dir / f"{split_name}_predictions.jsonl", batch_size=64)
        m = compute_metrics(preds)
        save_metrics(m, run_dir / f"{split_name}_metrics.json")
        rows[split_name] = m["accuracy"]
        print(f"  {run_name} / {split_name}: accuracy = {m['accuracy']:.2%}")
    return rows

scorer = ChoiceScorer(model, tok, max_tokens_per_batch=SCORER_TOKENS)
summary = {}
for k in BASELINE_SHOTS:
    summary[f"base {k}-shot"] = evaluate(scorer, f"{SHORT_NAME}_base_{k}shot", k)
pd.DataFrame(summary).T.style.format("{:.2%}")

## 6. LoRA fine-tuning

In [ ]:
import math
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments
from src.hf_model import build_train_examples, Collator, make_trainer_class, make_val_accuracy_callback

if DO_TRAIN:
    torch.manual_seed(SEED)
    if LOAD_IN_4BIT:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, task_type="CAUSAL_LM",
    ))
    model.print_trainable_parameters()

    train_examples = build_train_examples(tok, data["train"], max_len=MAX_LEN)
    steps = math.ceil(len(train_examples) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
    bf16 = torch.cuda.is_bf16_supported()

    args = TrainingArguments(
        output_dir=str(ROOT / "outputs" / "checkpoints"),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * steps)),
        logging_steps=10,
        save_strategy="no",                      # the callback saves the best adapter instead
        bf16=bf16, fp16=not bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )
    ft_scorer = ChoiceScorer(model, tok, max_tokens_per_batch=SCORER_TOKENS)
    val_cb = make_val_accuracy_callback(ft_scorer, data["val"], ADAPTER_DIR)
    trainer = make_trainer_class()(
        model=model, args=args, train_dataset=train_examples,
        data_collator=Collator(tok.pad_token_id), callbacks=[val_cb],
    )
    trainer.train()
    tok.save_pretrained(ADAPTER_DIR)

    pd.DataFrame(val_cb.history).to_csv(RESULTS_DIR / f"{SHORT_NAME}_lora_val_history.csv", index=False)
    print(pd.DataFrame(val_cb.history))

## 7. Evaluate the best fine-tuned adapter

In [ ]:
if DO_TRAIN:
    # The model in memory is from the last epoch; switch to the best epoch's adapter.
    model.load_adapter(str(ADAPTER_DIR), adapter_name="best")
    model.set_adapter("best")
    model.config.use_cache = True
    summary["LoRA 0-shot"] = evaluate(ft_scorer, f"{SHORT_NAME}_lora_0shot", 0)

final = pd.DataFrame(summary).T
final.to_csv(RESULTS_DIR / f"{SHORT_NAME}_summary.csv")
final.style.format("{:.2%}")

In [ ]:
# Per-subject test accuracy: baseline vs fine-tuned
def by_subject(run_name, split="test"):
    with open(RESULTS_DIR / run_name / f"{split}_metrics.json", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f)["by_subject"]).set_index("subject")["accuracy"]

runs = {f"base {k}-shot": f"{SHORT_NAME}_base_{k}shot" for k in BASELINE_SHOTS}
if DO_TRAIN:
    runs["LoRA 0-shot"] = f"{SHORT_NAME}_lora_0shot"
pd.DataFrame({name: by_subject(r) for name, r in runs.items()}).sort_values(list(runs)[-1], ascending=False).style.format("{:.2%}")

## 8. Save the adapter

The adapter is in `outputs/llama3.2-3b-instruct-lora/` (~100 MB). Colab deletes files when the session ends, so download it or copy it to Drive:

```python
from google.colab import drive; drive.mount("/content/drive")
!cp -r outputs/llama3.2-3b-instruct-lora /content/drive/MyDrive/
!cp -r results /content/drive/MyDrive/sinhala-mmlu-results
```

To load it again later:

```python
from peft import PeftModel
model, tok = load_model_and_tokenizer(MODEL_ID, load_in_4bit=True)
model = PeftModel.from_pretrained(model, "outputs/llama3.2-3b-instruct-lora")
scorer = ChoiceScorer(model, tok)
```